In [ ]:
# Section 1: Import required libraries
import os
import sys
import json
import subprocess
from pathlib import Path
from IPython.display import display, HTML

# Section 2: Resolve repository root and target script path
cwd = Path.cwd()
repo_root = cwd
# climb up until we find scripts/run_tests_noninteractive.py or stop at root
for _ in range(6):
    if (repo_root / 'scripts' / 'run_tests_noninteractive.py').exists():
        break
    if repo_root.parent == repo_root:
        break
    repo_root = repo_root.parent
script_path = repo_root / 'scripts' / 'run_tests_noninteractive.py'
if not script_path.exists():
    raise FileNotFoundError(f"Target script not found at expected path: {script_path}")

# Section 3: Detect workspace Python interpreter (VS Code settings or fallback)
python_exe = sys.executable
source = 'kernel'
vs_settings = repo_root / '.vscode' / 'settings.json'
if vs_settings.exists():
    try:
        s = json.loads(vs_settings.read_text())
        for key in ('python.defaultInterpreterPath','python.pythonPath'):
            p = s.get(key)
            if p:
                p = Path(p)
                if p.exists():
                    python_exe = str(p)
                    source = '.vscode'
                    break
    except Exception:
        pass
print(f"Using Python interpreter: {python_exe} (source={source})")

# Section 4: Run the script and capture stdout, stderr, and exit code
runs_dir = repo_root / 'runs'
runs_dir.mkdir(exist_ok=True)
stdout_file = runs_dir / 'last_run_stdout.txt'
stderr_file = runs_dir / 'last_run_stderr.txt'
exit_file = runs_dir / 'last_run_exit.json'

try:
    result = subprocess.run([python_exe, str(script_path)], cwd=str(repo_root), capture_output=True, text=True, timeout=300)
    out = result.stdout
    err = result.stderr
    code = result.returncode
except subprocess.TimeoutExpired as e:
    out = e.stdout or ''
    err = (e.stderr or '') + f"\nTIMEOUT: exceeded {e.timeout} seconds"
    code = 124
except Exception as e:
    out = ''
    err = str(e)
    code = 2

# Section 5: Persist and display captured outputs in notebook cells
stdout_file.write_text(out, encoding='utf-8')
stderr_file.write_text(err, encoding='utf-8')
exit_file.write_text(json.dumps({"exit_code": code}), encoding='utf-8')

print('=== EXIT CODE ===')
print(code)
print('\n=== STDOUT (first 2000 chars) ===')
print(out[:2000])
print('\n=== STDERR (first 2000 chars) ===')
print(err[:2000], file=sys.stderr)

display(HTML(f"<p>Saved stdout to <a href='../runs/last_run_stdout.txt'>last_run_stdout.txt</a></p><p>Saved stderr to <a href='../runs/last_run_stderr.txt'>last_run_stderr.txt</a></p><p>Saved exit code to <a href='../runs/last_run_exit.json'>last_run_exit.json</a></p>"))

# Section 6: Optional - run by inheriting terminal streams and compare
try:
    proc = subprocess.Popen([python_exe, str(script_path)], cwd=str(repo_root))
    proc.wait(timeout=10)
    code2 = proc.returncode
except Exception as e:
    code2 = None

print('\n=== EXIT CODE (subprocess inherit) ===')
print(code2)


# Expose variables for inspection
__captured = {'stdout': out, 'stderr': err, 'exit_code': code, 'exit_code_inherit': code2}
__captured


In [ ]:
# Additional runner cell: invoke the wrapper with sys.executable
import subprocess, sys
from pathlib import Path
wrap = Path(__file__).parent / '_run_wrapper.py'
if not wrap.exists():
    print('wrapper not found', file=sys.stderr)
else:
    print('Running wrapper with', sys.executable)
    r = subprocess.run([sys.executable, str(wrap)], capture_output=True, text=True)
    print('--- WRAPPER STDOUT ---')
    print(r.stdout)
    print('--- WRAPPER STDERR ---', file=sys.stderr)
    print(r.stderr, file=sys.stderr)
    print('--- WRAPPER EXIT CODE ---')
    print(r.returncode)
